# RAG with Chroma — Persistent Vector Store

Level up from FAISS — add built-in persistence, metadata filtering, and collection management to your RAG pipeline.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-chroma langchain-text-splitters chromadb fpdf2

---

## 1 · Build a Knowledge Base

Same pattern as Tutorial 06 — create sample documents, load, and split. This time we'll add **richer metadata** to demonstrate Chroma's filtering capabilities.

In [ ]:
from langchain_core.documents import Document

# ── Create documents with rich metadata for filtering demos ──
# In a real project, these come from loaders — we're creating them directly
# to have full control over metadata fields

documents = [
    Document(
        page_content="Self-attention computes Query, Key, and Value matrices from the input. "
        "The attention score is the dot product of Query and Key vectors, scaled by the "
        "square root of the dimension. Multi-head attention runs this in parallel across "
        "multiple subspaces, capturing different relationship types.",
        metadata={"source": "transformers.pdf", "topic": "architecture", "page": 1}
    ),
    Document(
        page_content="The Transformer encoder processes the full input sequence and produces "
        "contextual representations. The decoder generates output tokens one at a time, "
        "attending to both its own previous outputs and the encoder's representations "
        "via cross-attention.",
        metadata={"source": "transformers.pdf", "topic": "architecture", "page": 2}
    ),
    Document(
        page_content="GPT uses only the decoder stack for autoregressive generation. BERT "
        "uses only the encoder for bidirectional understanding. T5 uses both encoder "
        "and decoder for sequence-to-sequence tasks.",
        metadata={"source": "transformers.pdf", "topic": "models", "page": 3}
    ),
    Document(
        page_content="LoRA (Low-Rank Adaptation) freezes original weights and injects small "
        "trainable matrices into each layer. This reduces trainable parameters by 90%+ "
        "while maintaining performance. QLoRA combines LoRA with 4-bit quantization.",
        metadata={"source": "fine_tuning.pdf", "topic": "training", "page": 1}
    ),
    Document(
        page_content="Full fine-tuning updates all model parameters, requiring significant "
        "GPU memory. For a 7B parameter model, full fine-tuning needs ~28GB VRAM. "
        "LoRA reduces this to ~8GB, and QLoRA to ~4GB.",
        metadata={"source": "fine_tuning.pdf", "topic": "training", "page": 2}
    ),
    Document(
        page_content="RAG combines retrieval with generation. The model retrieves relevant "
        "documents from a knowledge base and uses them as context. This reduces "
        "hallucinations and supports knowledge updates without retraining.",
        metadata={"source": "rag_guide.pdf", "topic": "retrieval", "page": 1}
    ),
    Document(
        page_content="Vector databases store embedding vectors and support fast similarity "
        "search. FAISS is in-memory and optimized for speed. ChromaDB provides persistence "
        "and metadata filtering. Pinecone is a managed cloud solution.",
        metadata={"source": "rag_guide.pdf", "topic": "retrieval", "page": 2}
    ),
    Document(
        page_content="Embedding models convert text into dense vectors that capture semantic "
        "meaning. OpenAI's text-embedding-3-small produces 1536-dimensional vectors. "
        "Sentence-transformers/all-MiniLM-L6-v2 produces 384-dimensional vectors.",
        metadata={"source": "rag_guide.pdf", "topic": "embeddings", "page": 3}
    ),
]

print(f"Created {len(documents)} documents")
print(f"Sources: {set(d.metadata['source'] for d in documents)}")
print(f"Topics: {set(d.metadata['topic'] for d in documents)}")

---

## 2 · Create a Persistent Chroma Vector Store

**The Problem** — FAISS is in-memory. You have to manually save and reload. Chroma handles persistence automatically.

**The Solution** — Pass `persist_directory` to Chroma. Documents are written to disk and automatically reloaded on next initialization.

> FAISS is a filing cabinet you have to lock and unlock manually. Chroma is a database that auto-saves — you just open it and your data is there.

In [ ]:
import shutil, os

# Clean up any previous run (for reproducibility)
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Cleaned up previous Chroma DB")

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Create a persistent vector store ──
vectorstore = Chroma.from_documents(
    documents=documents,                  # our Document objects with metadata
    embedding=embeddings,                 # embedding model
    persist_directory="./chroma_db",       # auto-persists to this directory
    collection_name="ml_knowledge",        # named collection within the DB
)

print(f"Vector store created: {vectorstore._collection.count()} documents")
print(f"Collection: {vectorstore._collection.name}")
print(f"Persisted to: ./chroma_db/")

In [ ]:
# ── Reload from disk — no re-embedding needed ──
# This is what you'd do on subsequent runs

loaded_store = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings,         # must match the model used to create it
    collection_name="ml_knowledge",
)

print(f"Loaded from disk: {loaded_store._collection.count()} documents")

# Quick verify — search still works
results = loaded_store.similarity_search("attention", k=2)
for doc in results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:80]}...")

### What Just Happened?

- `from_documents()` embedded all documents and stored them in a Chroma collection on disk
- On subsequent runs, `Chroma(persist_directory=...)` loads the existing collection — no re-embedding
- No `save_local()` / `load_local()` dance like FAISS — persistence is automatic

---

## 3 · Metadata Filtering — Scoped Search

**The Problem** — Similarity search returns the closest vectors globally. Sometimes you only want results from a specific source, topic, or page range.

**The Solution** — Chroma's `filter` parameter applies metadata conditions before similarity search.

> Instead of searching the entire library, metadata filtering lets you say "only search the ML shelf" or "only search documents from 2024."

In [ ]:
# ── Filter by exact match on source ──
results = vectorstore.similarity_search(
    "How does attention work?",
    k=3,
    filter={"source": "transformers.pdf"}    # only search within this source
)

print("Filter: source = transformers.pdf\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] source={doc.metadata['source']}, page={doc.metadata['page']}")
    print(f"      {doc.page_content[:100]}...\n")

In [ ]:
# ── Filter by topic ──
results = vectorstore.similarity_search(
    "training large models efficiently",
    k=3,
    filter={"topic": "training"}
)

print("Filter: topic = training\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] topic={doc.metadata['topic']}, source={doc.metadata['source']}")
    print(f"      {doc.page_content[:100]}...\n")

In [ ]:
# ── Filter with comparison operators ──
results = vectorstore.similarity_search(
    "transformer architecture",
    k=3,
    filter={"page": {"$gte": 2}}            # page >= 2
)

print("Filter: page >= 2\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] page={doc.metadata['page']}, source={doc.metadata['source']}")
    print(f"      {doc.page_content[:100]}...\n")

In [ ]:
# ── Combine multiple filters with $and / $or ──
results = vectorstore.similarity_search(
    "model training",
    k=3,
    filter={
        "$or": [
            {"topic": "training"},
            {"topic": "architecture"}
        ]
    }
)

print("Filter: topic = training OR architecture\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] topic={doc.metadata['topic']}, source={doc.metadata['source']}")
    print(f"      {doc.page_content[:80]}...\n")

### What Just Happened?

- `filter={"key": "value"}` applies exact match filtering before similarity search
- Comparison operators: `$gte` (>=), `$gt` (>), `$lte` (<=), `$lt` (<), `$ne` (!=)
- Logical operators: `$and`, `$or` for combining multiple conditions
- Filters narrow the search space, then similarity search runs on the filtered subset

> **When to use:** Multi-source knowledge bases, role-based access, date-range queries, or any time you need to scope retrieval to a document subset.

---

## 4 · CRUD Operations — Managing Documents

**The Problem** — Knowledge bases evolve. Documents get updated, corrected, or deprecated.

**The Solution** — Chroma supports add, update, and delete operations by document ID.

> FAISS requires rebuilding the entire index for changes. Chroma lets you surgically modify individual documents.

In [ ]:
# ── Add new documents to an existing collection ──

new_docs = [
    Document(
        page_content="Prompt engineering is the practice of designing effective prompts to guide "
        "LLM behavior. Techniques include few-shot examples, chain-of-thought reasoning, "
        "and structured output formatting.",
        metadata={"source": "prompting.pdf", "topic": "prompting", "page": 1}
    ),
]

# Add with explicit IDs for later management
vectorstore.add_documents(new_docs, ids=["prompt_eng_001"])
print(f"After adding: {vectorstore._collection.count()} documents")

In [ ]:
# ── Update an existing document ──

updated_doc = Document(
    page_content="Prompt engineering is the practice of designing inputs that reliably produce "
    "desired LLM outputs. Key techniques: few-shot examples, chain-of-thought, "
    "self-consistency, and tree-of-thought reasoning.",
    metadata={"source": "prompting.pdf", "topic": "prompting", "page": 1}
)

vectorstore.update_documents(ids=["prompt_eng_001"], documents=[updated_doc])
print("Updated document prompt_eng_001")

# Verify the update
results = vectorstore.similarity_search("prompt engineering techniques", k=1)
print(f"Retrieved: {results[0].page_content[:100]}...")

In [ ]:
# ── Delete a document by ID ──

print(f"Before delete: {vectorstore._collection.count()} documents")

vectorstore.delete(ids=["prompt_eng_001"])

print(f"After delete:  {vectorstore._collection.count()} documents")

### What Just Happened?

- `add_documents()` with explicit `ids` lets you track and manage individual documents
- `update_documents()` replaces a document's content and re-embeds it — same ID, new content
- `delete()` removes documents by ID — no need to rebuild the index
- All changes auto-persist to disk

---

## 5 · RAG Chain with Chroma

Same LCEL pattern as Tutorial 06 — the only difference is the retriever source.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── Retriever from Chroma ──
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8}
)

# ── RAG prompt ──
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context. If the context doesn't
contain enough information, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── The RAG chain ──
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain with Chroma ready.")

In [ ]:
# ── Test the RAG chain ──

questions = [
    "How does self-attention work?",
    "What is the difference between LoRA and full fine-tuning?",
    "What are the benefits of RAG?",
    "How do I bake a chocolate cake?",  # not in knowledge base
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")
    print("-" * 70 + "\n")

---

## 6 · Filtered RAG — Scoped Retrieval in a Chain

The real power of Chroma + RAG — restrict the retriever to a specific document subset.

In [ ]:
# ── Retriever that ONLY searches within fine_tuning.pdf ──
filtered_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"source": "fine_tuning.pdf"}
    }
)

# ── RAG chain scoped to fine-tuning docs only ──
filtered_rag = (
    {"context": filtered_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# This question is about attention (in transformers.pdf) — filtered retriever won't find it
print("Filtered RAG (source=fine_tuning.pdf only):\n")
print(f"Q: What is self-attention?")
print(f"A: {filtered_rag.invoke('What is self-attention?')}")
print()
print(f"Q: How much VRAM does LoRA need?")
print(f"A: {filtered_rag.invoke('How much VRAM does LoRA need?')}")

### What Just Happened?

- The filtered retriever only searches within `fine_tuning.pdf` documents
- The self-attention question correctly returned "not enough information" — that topic is in `transformers.pdf`, which was excluded
- The LoRA question found the right answer because it's in `fine_tuning.pdf`

> **Key insight:** Filtered retrieval is essential for multi-tenant apps, role-based access, or when different questions should search different document subsets.

---

## 7 · Multiple Collections

Chroma supports multiple named collections in the same database — useful for separating different knowledge domains.

In [ ]:
# ── Create a second collection in the same database ──

company_docs = [
    Document(page_content="Our company uses a microservices architecture deployed on Kubernetes.",
             metadata={"dept": "engineering"}),
    Document(page_content="The Q3 revenue target is $5M with a 20% growth goal.",
             metadata={"dept": "sales"}),
    Document(page_content="All employees are eligible for 20 days of PTO per year.",
             metadata={"dept": "hr"}),
]

company_store = Chroma.from_documents(
    documents=company_docs,
    embedding=embeddings,
    persist_directory="./chroma_db",        # same directory
    collection_name="company_knowledge",     # different collection name
)

print(f"Collection 'ml_knowledge':      {vectorstore._collection.count()} docs")
print(f"Collection 'company_knowledge': {company_store._collection.count()} docs")
print(f"\nTwo collections, one database, completely isolated searches.")

In [ ]:
# ── Search each collection independently ──

print("Searching 'ml_knowledge' for 'attention':")
for doc in vectorstore.similarity_search("attention", k=1):
    print(f"  → {doc.page_content[:80]}...")

print("\nSearching 'company_knowledge' for 'vacation policy':")
for doc in company_store.similarity_search("vacation policy", k=1):
    print(f"  → {doc.page_content[:80]}...")

### What Just Happened?

- Two named collections (`ml_knowledge`, `company_knowledge`) live in the same `persist_directory`
- Each collection has its own documents and search index — completely isolated
- Use case: separate knowledge domains (engineering docs vs HR policies vs product specs) in the same app

---

## Key Takeaways

| Feature | FAISS (Tutorial 06) | Chroma (This Tutorial) |
|---------|--------------------|-----------------------|
| Persistence | Manual `save_local()` | Auto via `persist_directory` |
| Metadata Filtering | Not supported | `filter={"key": "value"}` |
| CRUD | Rebuild entire index | `add / update / delete` by ID |
| Collections | One index per file | Multiple named collections |
| Speed | Fastest (C++) | Fast (good for most cases) |
| Best For | Prototyping, read-only | Apps needing persistence + filtering |

**Practical advice:** Start with FAISS for quick experiments. Move to Chroma when you need persistence, filtering, or document management. Move to Pinecone/Weaviate when you need managed cloud infrastructure at scale.

**Next →** [08 · Memory](../08-memory/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*